# PyTorch Recommender Demo

This notebook is a lightweight companion to the main recommender example. Its purpose is to show practical familiarity with PyTorch on a small embedding-based recommender, not to build a production recommender.

## Setup

- Uses the same MovieLens 100k files as the main recommender notebook.
- Dependencies are managed through the repository `requirements.txt`.
- The model is a simple matrix-factorization style network with user and item embeddings.

## What This Demo Shows

- A compact PyTorch workflow for a simple embedding-based recommender.
- Explicit device placement so the notebook uses GPU if available and CPU otherwise.
- A time-aware `train -> validation -> test` split based on each eligible user's previous and latest favorable interactions (`rating >= 4`).
- Lightweight ranking training with sampled negatives and `Recall@10` evaluation.


In [77]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
pd.set_option("display.max_colwidth", 60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


## Data Loading

The notebook reuses `../data/u.data` and `../data/u.item`. If the files are missing, it downloads the official MovieLens 100k archive.

In [78]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

required_files = ["u.data", "u.item"]
if not all((data_dir / file_name).exists() for file_name in required_files):
    zip_path = data_dir / "ml-100k.zip"
    extract_dir = data_dir / "ml-100k"
    urlretrieve(MOVIELENS_URL, zip_path)
    with ZipFile(zip_path, "r") as zip_file:
        zip_file.extractall(data_dir)
    for file_name in required_files:
        source = extract_dir / file_name
        destination = data_dir / file_name
        if source.exists() and not destination.exists():
            destination.write_bytes(source.read_bytes())

ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item_id", "title"],
)

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings = ratings.merge(movies, on="item_id", how="left")
ratings.head()

,user_id,item_id,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)


## Preprocessing And Split

For consistency with the main notebook, we use a time-aware `train -> validation -> test` split. A user contributes to validation and test only if they have at least 3 total interactions, at least 2 favorable ratings (`rating >= 4`), and at least 1 interaction before the validation event.

Main slices used in this demo:
- `train_ratings`: earlier user history before the validation event
- `validation_ratings`: the second-latest favorable item per eligible user
- `test_ratings`: the latest favorable item per eligible user
- `train_positive`: favorable interactions from `train_ratings`
- `validation_positive` / `test_positive`: held-out favorable targets for ranking evaluation

How training examples are built:
- each observed favorable user-item interaction becomes a positive training example
- for the same user, we sample a few unseen items as negative training examples
- the model then learns to score the observed favorable item above those unseen items

This is why the deep demos use contrastive-style training while the main `TruncatedSVD` notebook does not: the neural ranking models need explicit positive and negative training examples, while `TruncatedSVD` learns from the interaction matrix directly.

This notebook uses contrastive-style training in a practical recommender sense: the model sees positive and negative user-item examples and learns to score the positives higher. It is simpler than formal contrastive learning setups, which usually compare paired representations directly and use specialized contrastive losses.

A more canonical modern ranking setup often uses pairwise losses or sampled-softmax / InfoNCE-style objectives rather than plain binary labels on sampled pairs. Those approaches are often stronger in larger systems, but this lighter pointwise setup is still common and defensible when the goal is a readable demo with a ranking objective.

Deep models train over many epochs, so they need a validation slice to monitor generalization during training. We use validation loss for early stopping, while `Recall@10` on held-out positives remains the ranking metric we care about.


In [79]:
POSITIVE_RATING = 4
MIN_USER_INTERACTIONS = 3
MIN_POSITIVE_INTERACTIONS = 2
NEGATIVES_PER_POSITIVE = 4
TOP_K = 10

user_ids = np.sort(ratings["user_id"].unique())
item_ids = np.sort(ratings["item_id"].unique())
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}

ratings["user_idx"] = ratings["user_id"].map(user_to_index)
ratings["item_idx"] = ratings["item_id"].map(item_to_index)


def make_time_split(frame, positive_rating=4, min_user_interactions=3, min_positive_interactions=2):
    frame = frame.sort_values(["user_id", "timestamp", "item_id"]).copy()
    train_parts = []
    validation_rows = []
    test_rows = []

    for _, user_history in frame.groupby("user_id", sort=False):
        positive_history = user_history[user_history["rating"] >= positive_rating]
        enough_history = len(user_history) >= min_user_interactions
        enough_positive = len(positive_history) >= min_positive_interactions

        if not (enough_history and enough_positive):
            train_parts.append(user_history)
            continue

        validation_row = positive_history.iloc[-2]
        test_row = positive_history.iloc[-1]
        train_end_idx = user_history.index.get_loc(validation_row.name)
        has_prior_history = train_end_idx > 0

        if not has_prior_history:
            train_parts.append(user_history)
            continue

        train_history = user_history.iloc[:train_end_idx]
        train_parts.append(train_history)
        validation_rows.append(validation_row)
        test_rows.append(test_row)

    train = pd.concat(train_parts, ignore_index=True)
    validation = pd.DataFrame(validation_rows).reset_index(drop=True)
    test = pd.DataFrame(test_rows).reset_index(drop=True)
    return train, validation, test


train_ratings, validation_ratings, test_ratings = make_time_split(
    ratings,
    positive_rating=POSITIVE_RATING,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_positive_interactions=MIN_POSITIVE_INTERACTIONS,
)

train_positive = train_ratings.loc[train_ratings["rating"] >= POSITIVE_RATING].reset_index(drop=True)
validation_positive = validation_ratings.reset_index(drop=True)
test_positive = test_ratings.reset_index(drop=True)
user_seen_items = train_ratings.groupby("user_id")["item_id"].apply(set).to_dict()


def build_pointwise_pairs(positive_frame, seen_lookup, all_item_ids, negatives_per_positive=4, seed=42):
    rng = np.random.default_rng(seed)
    all_item_ids = np.asarray(all_item_ids)
    user_indices = []
    item_indices = []
    labels = []

    for row in positive_frame.itertuples(index=False):
        seen_items = seen_lookup.get(row.user_id, set())
        negative_pool = all_item_ids[~np.isin(all_item_ids, list(seen_items))]
        if len(negative_pool) == 0:
            continue

        user_indices.append(row.user_idx)
        item_indices.append(row.item_idx)
        labels.append(1.0)

        sampled_negatives = rng.choice(
            negative_pool,
            size=min(negatives_per_positive, len(negative_pool)),
            replace=False,
        )
        for negative_item_id in sampled_negatives:
            user_indices.append(row.user_idx)
            item_indices.append(item_to_index[int(negative_item_id)])
            labels.append(0.0)

    return (
        np.asarray(user_indices, dtype=np.int64),
        np.asarray(item_indices, dtype=np.int64),
        np.asarray(labels, dtype=np.float32),
    )


train_user_idx, train_item_idx, train_labels = build_pointwise_pairs(
    train_positive,
    user_seen_items,
    item_ids,
    negatives_per_positive=NEGATIVES_PER_POSITIVE,
    seed=42,
)
validation_user_idx, validation_item_idx, validation_labels = build_pointwise_pairs(
    validation_positive,
    user_seen_items,
    item_ids,
    negatives_per_positive=NEGATIVES_PER_POSITIVE,
    seed=43,
)

print(f"Positive rating threshold: {POSITIVE_RATING}+")
print(f"Minimum total interactions for validation/test: {MIN_USER_INTERACTIONS}")
print(f"Minimum favorable interactions for validation/test: {MIN_POSITIVE_INTERACTIONS}")
print(f"Training positives: {len(train_positive):,}")
print(f"Validation positives: {len(validation_positive):,}")
print(f"Test positives: {len(test_positive):,}")
print(f"Training pairs after negative sampling: {len(train_labels):,}")
print(f"Validation pairs after negative sampling: {len(validation_labels):,}")


Positive rating threshold: 4+
Minimum total interactions for validation/test: 3
Minimum favorable interactions for validation/test: 2
Training positives: 53,491
Validation positives: 942
Test positives: 942
Training pairs after negative sampling: 267,455
Validation pairs after negative sampling: 4,710


## Model

We use a simple dot-product recommender with user and item bias terms. The model outputs a ranking logit: higher scores mean the model believes the user is more likely to engage with the item. For light regularization, PyTorch uses optimizer-level `weight_decay`, while the TensorFlow demo uses layer-level `L2` penalties. The exact implementation differs by framework, but the regularization intent is the same.

Compact model view:
- ranking logit: `s(u, i) = p_u^T q_i + b_u + b_i`
- probability used by the binary loss: `sigma(s(u, i))`
- regularization idea: keep embedding and bias weights from growing too large

Legend:
- `u`: user
- `i`: item
- `p_u`, `q_i`: user and item embedding vectors
- `b_u`, `b_i`: user and item bias terms
- `s(u, i)`: ranking logit before converting it into a probability
- `sigma(.)`: sigmoid function
- `batch`: the number of examples processed together in one training step
- `d`: embedding dimension

Why there is no explicit flatten step here: with 1D index tensors, PyTorch embedding lookups already return tensors shaped like `(batch, d)`, so the dot product can be computed directly. Regularization is applied through optimizer-level `weight_decay`.


Why this differs from the main notebook: the main flow uses `TruncatedSVD`, a classical matrix-factorization baseline that learns from the interaction matrix directly and does not need sampled positive/negative training. These deep-learning demos use sampled positive/negative pairs because that is a more natural way to train a lightweight neural ranking model for `Recall@10`.


In [80]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

    def forward(self, user_idx, item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        dot_product = (user_vec * item_vec).sum(dim=1)
        user_bias = self.user_bias(user_idx).squeeze(-1)
        item_bias = self.item_bias(item_idx).squeeze(-1)
        return dot_product + user_bias + item_bias

n_users = len(user_ids)
n_items = len(item_ids)
embedding_dim = 32
model = MatrixFactorization(n_users, n_items, embedding_dim).to(device)
model


MatrixFactorization(
  (user_embedding): Embedding(943, 32)
  (item_embedding): Embedding(1682, 32)
  (user_bias): Embedding(943, 1)
  (item_bias): Embedding(1682, 1)
)

## Training

We allow up to 100 epochs and stop early based on validation performance. Here, `train_loss` is training binary cross-entropy on sampled positive/negative pairs, and `val_loss` is the same metric on the validation slice.

Stopping rule: train for at most 100 epochs, monitor `val_loss`, keep the best model state seen so far, and stop early if `val_loss` fails to improve for 5 consecutive epochs. That 5-epoch span is the patience window. So the run ends either because validation stops improving within that window or because it reaches the maximum epoch limit.

We train in batches so the model can update from many user-item examples at once. Smaller batches are noisier and cheaper, which can sometimes help generalization, while larger batches are smoother and more efficient but use more memory.


PyTorch training loop pattern:

| PyTorch step | Meaning | TensorFlow mental equivalent |
|---|---|---|
| `optimizer.zero_grad()` | clear old gradients | usually handled inside `GradientTape` |
| `logits = model(...)` | forward pass | `model(x)` |
| `loss = loss_fn(...)` | compute error | same idea |
| `loss.backward()` | compute gradients | `tape.gradient(...)` |
| `optimizer.step()` | update weights | `optimizer.apply_gradients(...)` |

After `optimizer.step()`, the loop repeats from the top for the next batch. Once every batch is processed, the notebook runs validation and then starts the next epoch.


Mini-batch update intuition:

Within one epoch, the weights update after every batch, not only at the end of the epoch.

- Start with weights `W0`
- Batch 1: compute loss with `W0` -> compute gradients -> update to `W1`
- Batch 2: compute loss with `W1` -> compute gradients -> update to `W2`
- Batch 3: compute loss with `W2` -> compute gradients -> update to `W3`

So each batch builds on the previous update. This is why the notebook is using mini-batch gradient descent rather than full-batch gradient descent.

Quick comparison:

| Update style | When weights update |
|---|---|
| Full batch | once per epoch |
| Mini-batch (this notebook) | once per batch |
| SGD (`batch_size = 1`) | once per sample |


PyTorch-specific implementation notes:

- **Device handling (GPU / CPU):** in PyTorch, you manually move batches to the selected device with `.to(device)` before the forward pass.
- **Validation phase:** `model.eval()` switches the model into evaluation mode, and `torch.no_grad()` turns off gradient tracking so validation runs faster and uses less memory. In Keras, this is usually handled for you inside `fit()`.
- **Tracking metrics:** `train_loss = np.mean(epoch_losses)` and `val_loss = np.mean(validation_losses)` play the same role as `history.history["loss"]` and `history.history["val_loss"]` in Keras.
- **Early stopping:** the notebook implements the same idea as `tf.keras.callbacks.EarlyStopping`, but explicitly in Python by monitoring `val_loss` and counting epochs without improvement.
- **Saving and restoring the best weights:** `detach()` removes tensors from the computation graph, `cpu()` moves them off GPU, and `clone()` makes a clean copy so later updates do not mutate the saved snapshot. After training finishes, `model.load_state_dict(best_state)` restores that best validation state so downstream evaluation and recommendation steps use it rather than the last epoch by default.


In [81]:
train_dataset = TensorDataset(
    torch.tensor(train_user_idx, dtype=torch.long),
    torch.tensor(train_item_idx, dtype=torch.long),
    torch.tensor(train_labels, dtype=torch.float32),
)
train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)

validation_dataset = TensorDataset(
    torch.tensor(validation_user_idx, dtype=torch.long),
    torch.tensor(validation_item_idx, dtype=torch.long),
    torch.tensor(validation_labels, dtype=torch.float32),
)
validation_loader = DataLoader(validation_dataset, batch_size=2048, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
loss_fn = nn.BCEWithLogitsLoss()
history = []
best_val_loss = float("inf")
best_state = None
patience = 5
epochs_without_improvement = 0

for epoch in range(100):
    model.train()
    epoch_losses = []
    for user_batch, item_batch, label_batch in train_loader:
        # Move the current batch to CPU/GPU before the forward pass.
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        label_batch = label_batch.to(device)
        optimizer.zero_grad()
        logits = model(user_batch, item_batch)
        loss = loss_fn(logits, label_batch)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.item()))

    model.eval()
    validation_losses = []
    # Evaluation mode disables training-only behavior such as dropout.
    # Validation runs without gradient tracking because we are only measuring performance.
    with torch.no_grad():
        for user_batch, item_batch, label_batch in validation_loader:
            user_batch = user_batch.to(device)
            item_batch = item_batch.to(device)
            label_batch = label_batch.to(device)
            logits = model(user_batch, item_batch)
            validation_losses.append(float(loss_fn(logits, label_batch).item()))

    # Aggregate batch losses into one training and validation summary per epoch.
    train_loss = np.mean(epoch_losses)
    val_loss = np.mean(validation_losses)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save a clean snapshot of the best-performing parameter state for later restore.
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

if best_state is not None:
    # Restore the best validation state before downstream evaluation and recommendation.
    model.load_state_dict(best_state)

pd.DataFrame(history)


,epoch,train_loss,val_loss
0,1,2.365132,2.307254
1,2,2.196558,2.192408
2,3,2.040298,2.082504
3,4,1.893305,1.978751
4,5,1.755974,1.879878
...,...,...,...
95,96,0.158321,0.357843
96,97,0.157442,0.357371
97,98,0.156485,0.356787
98,99,0.155515,0.356579


## Evaluation

We report `Recall@10` on held-out favorable items. With one held-out favorable item per user, `Recall@10` is simply the share of evaluated users whose held-out item appears in the top 10 recommendations.


Evaluation note: for each user, the notebook scores the full candidate item set in one vectorized pass, masks items already seen in history, and then checks whether the held-out favorable item appears in the top `K`.


Quick reading guide: if `train_loss` and `val_loss` stay high while `Recall@10` stays low, the model may be underfitting. If training loss keeps falling but validation loss stops improving and `Recall@10` stays weak, the model may be overfitting. The healthiest pattern is lower validation loss together with stronger `Recall@10`.


In [82]:
def score_all_items_for_user(user_id):
    user_idx = user_to_index[user_id]
    # `torch.arange(...)` creates the full item index list, which means items = [0, 1, 2, ...].
    # `torch.full(...)` repeats the same user id, which means user = [u, u, u, ...].
    # Together, the model scores (u, 0), (u, 1), (u, 2), ... in one pass.
    item_tensor = torch.arange(n_items, dtype=torch.long, device=device)
    user_tensor = torch.full((n_items,), fill_value=user_idx, dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        scores = model(user_tensor, item_tensor).cpu().numpy()
    return scores


def recall_at_k(eval_frame, seen_frame, k=10):
    seen_lookup = seen_frame.groupby("user_id")["item_idx"].apply(set).to_dict()
    hits = []

    for row in eval_frame.itertuples(index=False):
        scores = score_all_items_for_user(row.user_id)
        seen_indices = np.fromiter(seen_lookup.get(row.user_id, set()), dtype=np.int64)
        if len(seen_indices) > 0:
            scores[seen_indices] = -np.inf

        top_indices = np.argsort(scores)[::-1][:k]
        hits.append(int(row.item_idx in top_indices))

    return float(np.mean(hits))


validation_recall = recall_at_k(validation_positive, train_ratings, k=TOP_K)
test_history = pd.concat([train_ratings, validation_ratings], ignore_index=True)
test_recall = recall_at_k(test_positive, test_history, k=TOP_K)

print(f"Validation Recall@{TOP_K}: {validation_recall:.4f}")
print(f"Test Recall@{TOP_K}: {test_recall:.4f}")


Validation Recall@10: 0.1200
Test Recall@10: 0.0892


Interpretation: higher `Recall@10` means the held-out favorable item appears more often near the top of the recommendation list. This is a ranking metric, so score ordering matters more than score calibration.


## Example Recommendations

Below we score all unseen movies for one user and return the top recommendations.

In [83]:
movie_lookup = movies.set_index("item_id")

def recommend_for_user(user_id, seen_frame, k=10):
    scores = score_all_items_for_user(user_id)
    seen_indices = seen_frame.loc[seen_frame["user_id"] == user_id, "item_idx"].to_numpy()
    scores[seen_indices] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]
    top_item_ids = [item_ids[idx] for idx in top_indices]
    return pd.DataFrame(
        {
            "item_id": top_item_ids,
            "title": movie_lookup.loc[top_item_ids, "title"].values,
            "ranking_score": scores[top_indices],
        }
    )

example_user_id = int(test_positive.iloc[0]["user_id"])
example_seen = pd.concat([train_ratings, validation_ratings], ignore_index=True)
recommend_for_user(example_user_id, example_seen, k=10)


,item_id,title,ranking_score
0,511,Lawrence of Arabia (1962),4.495220
1,515,"Boot, Das (1981)",4.072537
2,273,Heat (1995),3.965307
3,475,Trainspotting (1996),3.445547
4,474,Dr. Strangelove or: How I Learned to Stop Worrying and L...,3.287080
5,652,Rosencrantz and Guildenstern Are Dead (1990),2.822466
6,455,Jackie Chan's First Strike (1996),2.766380
7,513,"Third Man, The (1949)",2.664104
8,566,Clear and Present Danger (1994),2.432741
9,408,"Close Shave, A (1995)",2.416473


## What This Demo Does Not Cover

- Production-scale candidate generation, ranking, and re-ranking.
- Rich side features such as genres, user metadata, or sequence context.
- ANN retrieval, online testing, or production monitoring.
- A deeper comparison against stronger recommender baselines.
